# Concurrent dual-sensor fusion: a Pyramid + ZWFS observing one shared DM

This notebook trains a single fusion reconstructor that is fed **simultaneously** by two structurally different WFS -- a modulated Pyramid (700 nm) and a ZWFS (1100 nm) -- both sensing the *same* residual OPD at the *same* tick rate, driving *one* shared DM. This is a genuinely different architecture from `TwoStageAO.ipynb`: that notebook chains two WFS/DM pairs sequentially (one frozen, treated as a slow input to the other); here both sensors are live and synchronous, feeding one reconstructor, one optimizer, and one `M2C`. See `Ideas/04-concurrent-dual-sensor-fusion.md` for the full motivation.

**Architecture:**
- One shared atmosphere/telescope feeds both sensors every tick.
- `wfs_a` (Pyramid, 700 nm) and `wfs_b` (ZWFS, 1100 nm) both sense the identical residual OPD -- the residual left over from the *same* shared DM's own previous correction -- every tick, not a cascade.
- Each sensor is preprocessed by its own `FramePreprocess` instance (different pupil geometry/count), producing `(B,4,H,W)` and `(B,1,H,W)` pupil-image stacks that are concatenated into one `(B,5,H,W)` input for the fusion CNN.
- There is one leaky integrator, one `loop_gain`/`loop_leak` draw, one shared DM -- a single control decision per tick, not two separate loops.

**Training strategy:** `AI4AO.Trainer.Trainer` only supports a single WFS/DM pair, so it can't run the fused loop directly (both sensors are live every tick, unlike `TwoStageAO`'s frozen-slow-stage-as-dataset trick, which only works because one stage is frozen and runs at a *different* rate). So this notebook trains three reconstructors under otherwise identical conditions (same shared `dm`/`M2C`, same atmosphere distribution, same loss):

1. **Baseline A** -- Pyramid alone drives the shared DM. A completely standard single-WFS closed loop, trained with the ordinary `Trainer`.
2. **Baseline B** -- ZWFS alone drives the shared DM. Same, with `Trainer` pointed at `wfs_b` instead.
3. **Fused** -- both sensors drive the shared DM together, every tick. This needs a hand-rolled training loop (defined in a cell below, modeled directly on `Trainer.train()`'s internals) since `Trainer` can't take two WFS.

The point of training all three is the ablation: "fusion beats either sensor alone" is only a real claim if we can compare it against both single-sensor baselines trained the same way -- a single fused-loop residual number on its own wouldn't demonstrate that.

In [ ]:
from mmengine import Config
import matplotlib.pyplot as plt
import numpy as np
import torch
import torch.nn as nn
import os
from tqdm import tqdm

from AI4AO import PyramidWFS, ZernikeWFS, PhaseDataset, FramePreprocess, DeformableMirror, Trainer, imshow_multiple
from AI4AO.LossFunctions import LogResidualVarianceLoss

device = 'cuda'  # set to "cpu" if CUDA is not available

## Loading the shared configuration

`ConcurrentDualSensorFusion_params.py` holds one shared `WFSParams` (telescope + detector-noise + frame-preprocessing settings common to both sensors), one shared `AtmosParams`/`LoopParams`/`DMParams` (a single tick rate, a single shared DM -- unlike `TwoStageAO`, there is no per-stage r0 rescale or speed ratio to derive, since both sensors observe the identical physical OPD concurrently), and `TrainParams`.

We copy `WFSParams` once per sensor and add the sensor-specific keys ourselves: `Wavelength`/`Modulation` for the Pyramid, `Wavelength`/`MaskType`/`Use_MTF`/`MTF_upscale` for the ZWFS (`MaskType: "zwfs"` is the single-mask code path in `AI4AO/ZernikeWFS.py`, giving exactly one pupil image -- as opposed to `"vzwfs"`'s two masks used in `TwoStageAO`).

In [ ]:
paramfile = 'DualSensorFusion_params.py'

AtmosParams = Config.fromfile(paramfile)['AtmosParams']
WFSParams = Config.fromfile(paramfile)['WFSParams']
LoopParams = Config.fromfile(paramfile)['LoopParams']
DMParams = Config.fromfile(paramfile)['DMParams']
TrainParams = Config.fromfile(paramfile)['TrainParams']

# wfs_a: modulated Pyramid at 700nm
WFSParams_a = WFSParams.copy()
WFSParams_a.update(Wavelength=700e-9, Modulation=3)

# wfs_b: ZWFS at 1100nm (MaskType="zwfs" -> a single phase-shifting dot, 1 pupil image)
WFSParams_b = WFSParams.copy()
WFSParams_b.update(Wavelength=1100e-9, MaskType="zwfs", Use_MTF=False, MTF_upscale=10)

# LogResidualVarianceLoss needs one wavelength to convert the shared residual OPD to
# phase; we use the ZWFS's 1100nm throughout (baselines and fused alike) so all three
# conditions' loss curves and residual-RMS numbers are computed on the same physical
# basis and are directly comparable.
loss_wavelength = WFSParams_b["Wavelength"]

PATH = "../../Data/ConcurrentDualSensorFusion/"
os.makedirs(PATH, exist_ok=True)

## The shared DM and command basis

Both sensors and all three trained reconstructors (baseline A, baseline B, fused) drive the exact same `dm`/`M2C` -- this is what makes the three training runs a fair, controlled ablation rather than three unrelated instruments. The DM is frozen (`.eval()` + `requires_grad_(False)`) throughout: only the reconstructors' weights are ever trained here.

`DMParams['Nmodes']` isn't known ahead of time (it depends on how many actuator-grid points fall inside the circular aperture cutoff for `Nactuator=15`, which `DeformableMirror.__init__` computes as `dm.totalAct`), so we read it back after construction and use it as the number of Zernike modes for `dm.MakeZernikeM2C()` -- a Zernike modal command basis (this repo has no true Karhunen-Loeve basis; `MakeZernikeM2C` is the modal-basis builder it provides), rather than the purely zonal identity basis `TwoStageAO` uses.

In [ ]:
dm = DeformableMirror(WFSParams_a, DMParams, device)
dm.eval()
dm.requires_grad_(False)

nModes = int(dm.totalAct.item()*0.7)
M2C = dm.MakeZernikeM2C(nModes=nModes)

# Shared by every Trainer/hand-rolled loop below: pinv(dm(M2C.T)) only depends on dm/M2C,
# so it is identical for baseline A, baseline B, and the fused loop.
z_inv = torch.linalg.pinv(dm(M2C.T).flatten(start_dim=-2))

In [ ]:
wfs_a = PyramidWFS(WFSParams_a, device)
wfs_a.eval()

wfs_b = ZernikeWFS(WFSParams_b, device)
wfs_b.eval()

framePreprocessor_a = FramePreprocess(WFSParams_a, wfs_a, device)
framePreprocessor_a.ProcessReference(wfs_a.reference_intensity)

framePreprocessor_b = FramePreprocess(WFSParams_b, wfs_b, device)
framePreprocessor_b.ProcessReference(wfs_b.reference_intensity)

print(f"wfs_a (Pyramid) pupils: {wfs_a.pupil_centers.shape[0]}, wfs_b (ZWFS) pupils: {wfs_b.pupil_centers.shape[0]}")

## Reconstructor architecture

One `PupilCNN` class, parametrized by input channel count, serves all three trained reconstructors: 4 channels for the Pyramid alone, 1 channel for the ZWFS alone, and 5 channels for the fused case (the Pyramid's 4 pupil images concatenated with the ZWFS's 1). Each pupil image is processed independently in the stem (`groups=n_channels`) before the encoder mixes information across channels -- the same single-shared-stem pattern `TwoStageAO.ipynb`'s `PWFSNet` uses, just generalized to 5 channels for the fusion case. `FramePreprocess.GetTrainingPupils`' output size only depends on the shared `WFSParams` keys (`Nres`, `Extract_pupils_pad`, `Bin_factor`), not on which WFS produced the frame, so `framePreprocessor_a`/`framePreprocessor_b`'s outputs are already the same `(H,W)` and concatenate directly along the channel dimension with no resizing needed.

In [ ]:
class PupilCNN(nn.Module):
    def __init__(self, n_channels, Nmodes):
        super().__init__()

        self.stem = nn.Sequential(
            # Process each pupil image independently
            nn.Conv2d(n_channels, 8 * n_channels, kernel_size=11, padding=5, groups=n_channels),
            nn.GELU(),

            nn.Conv2d(8 * n_channels, 16 * n_channels, kernel_size=7, padding=3, groups=n_channels),
            nn.GELU(),

            nn.MaxPool2d(2),
        )

        self.encoder = nn.Sequential(
            nn.Conv2d(16 * n_channels, 64, kernel_size=5, padding=2),
            nn.GELU(),

            nn.MaxPool2d(2),

            nn.Conv2d(64, 128, kernel_size=3, padding=1),
            nn.GELU(),

            nn.MaxPool2d(2),

            nn.Conv2d(128, 256, kernel_size=3, padding=1),
            nn.GELU(),

            nn.AdaptiveAvgPool2d(1),
        )

        self.head = nn.Sequential(
            nn.Flatten(),
            nn.Linear(256, Nmodes),
        )

    def forward(self, x):
        x = self.stem(x)
        x = self.encoder(x)
        return self.head(x)

## Baseline A: Pyramid alone

A completely standard single-WFS closed loop -- `wfs_a` drives the shared `dm` on its own, trained exactly like the single-stage `Tutorials/<Instrument>` notebooks, using the ordinary `Trainer`. This is one arm of the ablation: how well does the Pyramid do without any help from the ZWFS?

In [ ]:
dataset_a = PhaseDataset(WFSParams_a, AtmosParams, LoopParams, DMParams, device)
dataset_a.generateClosedLoop = True

phaseReconstructor_a = PupilCNN(n_channels=4, Nmodes=nModes).to(device=device)
total_params = sum(p.numel() for p in phaseReconstructor_a.parameters() if p.requires_grad)
print(f"Baseline A (Pyramid) reconstructor -- total trainable parameters: {total_params:,}")

optimizer_a = torch.optim.AdamW(phaseReconstructor_a.parameters(), TrainParams['lrn'], fused=True)
loss_a = LogResidualVarianceLoss(dataset_a.pupil, wavelength=loss_wavelength)

trainer_a = Trainer(wfs=wfs_a,
                     framePreprocessor=framePreprocessor_a,
                     dm=dm,
                     M2C=M2C,
                     phaseReconstructor=phaseReconstructor_a,
                     dataset=dataset_a,
                     loss=loss_a,
                     optimizer=optimizer_a)

try:
    trainer_a.load_checkpoint(PATH + "BaselinePyramidCNN.pth", load_optimizer=False)
except KeyError:
    print("Starting from scratch")

In [ ]:
loss_tracker_a, loss_tracker_a_ideal = trainer_a.train(TrainParams['TrainRunNb'], TrainParams['ClosedLoopIterations'])

In [ ]:
trainer_a.plot_losses(loss_tracker_a, loss_tracker_a_ideal)

In [ ]:
trainer_a.save_checkpoint(PATH + "BaselinePyramidCNN.pth")

## Baseline B: ZWFS alone

Same pattern, with `Trainer` pointed at `wfs_b`/`framePreprocessor_b` instead -- the other arm of the ablation.

In [ ]:
dataset_b = PhaseDataset(WFSParams_a, AtmosParams, LoopParams, DMParams, device)
dataset_b.generateClosedLoop = True

phaseReconstructor_b = PupilCNN(n_channels=1, Nmodes=nModes).to(device=device)
total_params = sum(p.numel() for p in phaseReconstructor_b.parameters() if p.requires_grad)
print(f"Baseline B (ZWFS) reconstructor -- total trainable parameters: {total_params:,}")

optimizer_b = torch.optim.AdamW(phaseReconstructor_b.parameters(), TrainParams['lrn'], fused=True)
loss_b = LogResidualVarianceLoss(dataset_b.pupil, wavelength=loss_wavelength)

trainer_b = Trainer(wfs=wfs_b,
                     framePreprocessor=framePreprocessor_b,
                     dm=dm,
                     M2C=M2C,
                     phaseReconstructor=phaseReconstructor_b,
                     dataset=dataset_b,
                     loss=loss_b,
                     optimizer=optimizer_b)

try:
    trainer_b.load_checkpoint(PATH + "BaselineZWFSCNN.pth", load_optimizer=False)
except KeyError:
    print("Starting from scratch")

In [ ]:
loss_tracker_b, loss_tracker_b_ideal = trainer_b.train(TrainParams['TrainRunNb'], TrainParams['ClosedLoopIterations'])

In [ ]:
trainer_b.plot_losses(loss_tracker_b, loss_tracker_b_ideal)

In [ ]:
trainer_b.save_checkpoint(PATH + "BaselineZWFSCNN.pth")

## Fused: both sensors drive the shared DM together

`wfs_a` and `wfs_b` now sense the *same* `residual_opd` every tick, are preprocessed independently, concatenated into one `(B,5,H,W)` tensor, and fed to one `PupilCNN(n_channels=5, ...)`. `Trainer` can't run this (it only takes one WFS), so the training step below is hand-rolled, mirroring `Trainer.train()`'s internals exactly -- same leaky-integrator bookkeeping, same `z_inv`-based `Ze`/ideal-loss diagnostics -- just propagating the residual through two WFS instead of one.

In [ ]:
dataset_fused = PhaseDataset(WFSParams_a, AtmosParams, LoopParams, DMParams, device)
dataset_fused.generateClosedLoop = True

phaseReconstructor_fused = PupilCNN(n_channels=5, Nmodes=nModes).to(device=device)
total_params = sum(p.numel() for p in phaseReconstructor_fused.parameters() if p.requires_grad)
print(f"Fused reconstructor -- total trainable parameters: {total_params:,}")

optimizer_fused = torch.optim.AdamW(phaseReconstructor_fused.parameters(), TrainParams['lrn'], fused=True)
loss_fused = LogResidualVarianceLoss(dataset_fused.pupil, wavelength=loss_wavelength)

In [ ]:
def train_fused(training_steps, closed_loop_iterations, dataset, phaseReconstructor, optimizer, loss):
    """Closed-loop training step for the fused dual-sensor loop, mirroring Trainer.train()
    but propagating the same residual_opd through wfs_a and wfs_b every iteration and
    feeding the reconstructor their concatenated, preprocessed pupil images."""
    dm.eval()
    wfs_a.eval()
    wfs_b.eval()

    loss_tracker = torch.zeros(training_steps // closed_loop_iterations, device=device)
    loss_tracker_ideal = torch.zeros(training_steps // closed_loop_iterations, device=device)

    phaseReconstructor.train()

    M2C_T = M2C.T

    progressBar = tqdm(range(training_steps // closed_loop_iterations))

    for u in progressBar:
        with torch.no_grad():
            batch = dataset[0]
            opd_gt = batch["opd"]
            pupilGT = batch["pupil"]
            gain = batch["loop_gain"]
            leak = batch["loop_leak"]
            photons = batch["nphotons"]
            ron = batch["ron"]

            wfs_a.SetPhotonsAndRON(photons, ron)
            wfs_b.SetPhotonsAndRON(photons, ron)

            z_estimated = torch.zeros(opd_gt.shape[0], nModes, device=device)
            z_buffer = torch.zeros_like(z_estimated)
            z_output = torch.zeros_like(z_estimated)
            opd_reconstructed = torch.zeros_like(opd_gt)

            total_loss = 0
            ideal_loss = 0

        for i in range(closed_loop_iterations):
            with torch.no_grad():
                if i > 0:
                    batch = dataset[i]
                    opd_gt = batch["opd"]
                    pupilGT = batch["pupil"]

                residual_opd = opd_gt - opd_reconstructed
                Ze = torch.matmul(residual_opd.flatten(start_dim=-2), z_inv)

                z_estimated = z_estimated * leak + gain * z_buffer
                z_buffer = torch.clone(z_output)

                wfs_a_frame = wfs_a(residual_opd, pupilGT)
                wfs_b_frame = wfs_b(residual_opd, pupilGT)
                preprocessed_a = framePreprocessor_a.ProcessFrame(wfs_a_frame)
                preprocessed_b = framePreprocessor_b.ProcessFrame(wfs_b_frame)
                fused_input = torch.cat([preprocessed_a, preprocessed_b], dim=1)

            z_output = phaseReconstructor(fused_input)

            opd_reconstructed = dm(z_estimated @ M2C_T)
            opd_reconstructed_iter = dm(z_output @ M2C_T)
            opd_reconstructed_iter_ideal = dm(Ze @ M2C_T)

            corrected_residual_opd = residual_opd - opd_reconstructed_iter
            total_loss = total_loss + loss(Ze, z_output, pupilGT, residual_opd, corrected_residual_opd, wfs_a_frame) / closed_loop_iterations

            with torch.no_grad():
                ideal_corrected_residual_opd = residual_opd - opd_reconstructed_iter_ideal
                ideal_loss = ideal_loss + loss(Ze, Ze, pupilGT, residual_opd, ideal_corrected_residual_opd, wfs_a_frame) / closed_loop_iterations

        optimizer.zero_grad(set_to_none=True)
        total_loss.backward()
        optimizer.step()

        loss_tracker[u] = total_loss.detach()
        loss_tracker_ideal[u] = ideal_loss.detach()

        if u % (300 // closed_loop_iterations) == 1:
            lower_lim = max(0, u - 100 // closed_loop_iterations)
            progressBar.set_postfix({'Loss': float(loss_tracker[lower_lim:u].mean()), 'Loss_ideal': float(loss_tracker_ideal[lower_lim:u].mean())})

    return loss_tracker, loss_tracker_ideal


def save_fused_checkpoint(path, model, optimizer):
    torch.save({
        "phase_reconstructor_state_dict": model.state_dict(),
        "optimizer_state_dict": optimizer.state_dict(),
    }, path)


def load_fused_checkpoint(path, model, optimizer, load_optimizer=True):
    if not os.path.exists(path):
        print(f"No checkpoint found at {path}, starting from scratch")
        return
    checkpoint = torch.load(path, map_location=device)
    model.load_state_dict(checkpoint["phase_reconstructor_state_dict"])
    if load_optimizer and "optimizer_state_dict" in checkpoint:
        optimizer.load_state_dict(checkpoint["optimizer_state_dict"])

In [ ]:
load_fused_checkpoint(PATH + "FusedCNN.pth", phaseReconstructor_fused, optimizer_fused, load_optimizer=False)

In [ ]:
loss_tracker_fused, loss_tracker_fused_ideal = train_fused(
    TrainParams['TrainRunNb'], TrainParams['ClosedLoopIterations'],
    dataset_fused, phaseReconstructor_fused, optimizer_fused, loss_fused
)

In [ ]:
trainer_a.plot_losses(loss_tracker_fused, loss_tracker_fused_ideal)  # reuses Trainer's plotting helper; it only touches the tensors passed in

In [ ]:
save_fused_checkpoint(PATH + "FusedCNN.pth", phaseReconstructor_fused, optimizer_fused)

## Ablation: does fusion actually beat either sensor alone?

To compare the three trained reconstructors fairly, we evaluate each of them on **identical atmosphere realizations**: a fresh `PhaseDataset` is built and `torch.manual_seed` is reset to the same value immediately before each rollout, so the wind/r0/noise draws (which come from the global RNG, see `conftest.py`'s `seed_rng` fixture / this repo's testing conventions) line up exactly across the three conditions. Baseline A/B reuse `Trainer.evaluate()`; the fused case needs its own no-grad rollout function, `evaluate_fused`, mirroring `Trainer.evaluate()` but with both WFS.

In [ ]:
@torch.no_grad()
def evaluate_fused(n_steps, dataset):
    dm.eval()
    wfs_a.eval()
    wfs_b.eval()
    phaseReconstructor_fused.eval()

    M2C_T = M2C.T

    batch = dataset[0]
    opd_gt = batch["opd"]
    pupilGT = batch["pupil"]
    gain = batch["loop_gain"]
    leak = batch["loop_leak"]
    photons = batch["nphotons"]
    ron = batch["ron"]

    wfs_a.SetPhotonsAndRON(photons, ron)
    wfs_b.SetPhotonsAndRON(photons, ron)

    z_estimated = torch.zeros(opd_gt.shape[0], nModes, device=device)
    z_buffer = torch.zeros_like(z_estimated)
    z_output = torch.zeros_like(z_estimated)
    opd_reconstructed = torch.zeros_like(opd_gt)

    residuals, wfs_a_frames, wfs_b_frames = [], [], []

    for i in range(n_steps):
        if i > 0:
            batch = dataset[i]
            opd_gt = batch["opd"]
            pupilGT = batch["pupil"]

        residual_opd = opd_gt - opd_reconstructed

        wfs_a_frame = wfs_a(residual_opd, pupilGT)
        wfs_b_frame = wfs_b(residual_opd, pupilGT)
        preprocessed_a = framePreprocessor_a.ProcessFrame(wfs_a_frame, False)
        preprocessed_b = framePreprocessor_b.ProcessFrame(wfs_b_frame, False)
        fused_input = torch.cat([preprocessed_a, preprocessed_b], dim=1)
        z_output = phaseReconstructor_fused(fused_input)

        z_buffer = torch.clone(z_output)
        if i > n_steps * 0.3:
            z_estimated = z_estimated * leak + gain * z_buffer

        opd_reconstructed = dm(z_estimated @ M2C_T)

        residuals.append(residual_opd)
        wfs_a_frames.append(wfs_a_frame)
        wfs_b_frames.append(wfs_b_frame)

    return torch.stack(residuals), torch.stack(wfs_a_frames), torch.stack(wfs_b_frames)


def residual_rms_nm(residual_opd, pupil, steady_state_frac=0.5):
    """RMS wavefront error (nm) over the pupil, averaged over the steady-state (post loop-closure) tail of the rollout."""
    n_steps = residual_opd.shape[0]
    tail = residual_opd[int(n_steps * steady_state_frac):]
    rms_per_step = torch.sqrt(torch.mean(tail[..., pupil.bool()] ** 2, dim=-1))
    return (rms_per_step.mean() * 1e9).item()

In [ ]:
n_frames = 100
seed = 1234

torch.manual_seed(seed)
eval_dataset_a = PhaseDataset(WFSParams_a, AtmosParams, LoopParams, DMParams, device)
eval_dataset_a.generateClosedLoop = True
result_a = trainer_a.evaluate(n_steps=n_frames, dataset=eval_dataset_a)
rms_a = residual_rms_nm(result_a.residual_opd, eval_dataset_a.pupil)

torch.manual_seed(seed)
eval_dataset_b = PhaseDataset(WFSParams_a, AtmosParams, LoopParams, DMParams, device)
eval_dataset_b.generateClosedLoop = True
result_b = trainer_b.evaluate(n_steps=n_frames, dataset=eval_dataset_b)
rms_b = residual_rms_nm(result_b.residual_opd, eval_dataset_b.pupil)

torch.manual_seed(seed)
eval_dataset_fused = PhaseDataset(WFSParams_a, AtmosParams, LoopParams, DMParams, device)
eval_dataset_fused.generateClosedLoop = True
residual_fused, wfs_a_frames_fused, wfs_b_frames_fused = evaluate_fused(n_frames, eval_dataset_fused)
rms_fused = residual_rms_nm(residual_fused, eval_dataset_fused.pupil)

print(f"Baseline A (Pyramid alone):  {rms_a:.1f} nm RMS")
print(f"Baseline B (ZWFS alone):     {rms_b:.1f} nm RMS")
print(f"Fused (Pyramid + ZWFS):      {rms_fused:.1f} nm RMS")

In [ ]:
fig, ax = plt.subplots()
labels = ["Pyramid alone", "ZWFS alone", "Fused"]
values = [rms_a, rms_b, rms_fused]
ax.bar(labels, values)
ax.set_ylabel("Steady-state residual wavefront error (nm RMS)")
ax.set_title("Concurrent dual-sensor fusion: ablation")
plt.show()

## Visualizing the fused closed loop

A quick sanity check of what each sensor actually sees in the fused system, at steady state (last frame of the rollout above).

In [ ]:
fig, axes = imshow_multiple(
    [
        {"tensor": residual_fused[-1], "title": "Residual OPD (fused, steady state)", "same_scale": True},
        {"tensor": wfs_a_frames_fused[-1], "title": "wfs_a (Pyramid) frame"},
        {"tensor": wfs_b_frames_fused[-1], "title": "wfs_b (ZWFS) frame"},
    ],
    max_channel_number=4
)
plt.show()